In [ ]:
# ---- EXPERIMENT CONFIG: edit this cell for each run --------------------------
# The defaults in pose3d/config.py are the reference recipe (6te3pvqa, 9.46 deg): Clifford Flow,
# pretrained ResNet-101, n_cond_mv=64, n_time_samples=8, hidden_dim 32, 100 epochs, lr 1e-4,
# 32-sample medoid evaluation. DDP over every visible GPU (Kaggle T4 x2) and the pre-built Pascal3D
# tensors are on by default too. So EXTRA only needs the flags that differ for this run, e.g.
#   ["--use_warp"]                          # turn an experimental feature on
#   ["--no-medoid_eval"]                    # turn an adopted feature off
#   ["--adapter_grid=9", "--adapter_channels=96"]
#   ["--encoder=resnet50"]                  # switch back to the smaller backbone
#   ["--no-ddp"]  /  ["--no-pre_cache"]     # turn the speed features off
# Attach the datasets syfry5suvzovvakmuj/pascal3d and syfry5suvzovvakmuj/pascal3d-ram-cache.
# W&B login. Leave None to read the `wandb_api_key` Kaggle secret (which Kaggle's API cannot attach, so
# a notebook pushed with `kaggle kernels push` fails at that step). For a PRIVATE notebook only, a
# launch script may set the key here instead; never commit a key, this file lives in a public repo.
WANDB_API_KEY = None
# Repo to clone (public, so no token is needed) and the branch to run.
REPO = 'optimist1938/GA-Research'
BRANCH = 'main'
RUN_NAME = 'clifford_flow_baseline'
EXTRA = []
ARGS = EXTRA


In [ ]:
import os, shutil, subprocess

if WANDB_API_KEY:
    wandb_api_key = WANDB_API_KEY
else:
    from kaggle_secrets import UserSecretsClient
    wandb_api_key = UserSecretsClient().get_secret("wandb_api_key")
assert wandb_api_key, "set WANDB_API_KEY in the first cell or attach the wandb_api_key secret"

CLONE_URL = f"https://github.com/{REPO}.git"
if os.path.exists("/kaggle/working/GA-Research"):
    shutil.rmtree("/kaggle/working/GA-Research")
os.chdir("/kaggle/working")
r = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, CLONE_URL],
                   capture_output=True, text=True)
print(r.stdout + r.stderr)
r.check_returncode()
print(subprocess.run(["git", "-C", "GA-Research", "log", "--oneline", "-3"],
                     capture_output=True, text=True).stdout)

In [ ]:
# No Poetry: keep Kaggle's preinstalled torch, numpy, ... (the constraints pin them) and install only
# what it lacks.
import importlib.metadata as md
core = ["torch", "torchvision", "numpy", "scipy", "pillow", "tqdm", "wandb", "scikit-image",
        "matplotlib", "pandas", "sympy", "astropy", "packaging"]
cons = []
for p in core:
    try:
        cons.append(f"{p}=={md.version(p)}")
    except md.PackageNotFoundError:
        pass
open("/kaggle/working/constraints.txt", "w").write("\n".join(cons) + "\n")
subprocess.run(["pip", "install", "-q", "-c", "/kaggle/working/constraints.txt", "e3nn==0.5.9", "healpy==1.19.0",
                "git+https://github.com/chagrygoris/image2sphere.git",
                "git+https://github.com/chagrygoris/clifford-group-equivariant-neural-networks.git"], check=True)
# GATr, only needed for --vector_field=gatr (--no-deps: its pins would replace torch).
subprocess.run(["pip", "install", "-q", "--no-deps", "einops", "opt_einsum",
                "git+https://github.com/Qualcomm-AI-research/geometric-algebra-transformer.git"], check=True)


In [ ]:
import wandb
wandb.login(key=wandb_api_key)


In [ ]:
import sys
os.chdir("/kaggle/working/GA-Research/pose3d")   # the package is found from here, no install needed

PASCAL = next(p for p in ["/kaggle/input/datasets/syfry5suvzovvakmuj/pascal3d",
                          "/kaggle/input/pascal3d"] if os.path.exists(p))
cmd = [sys.executable, "-u", "-m", "pose3d",
       f"--path_to_datasets={PASCAL}",
       "--platform=kaggle",
       f"--run_name={RUN_NAME}",
       *ARGS]
print(" ".join(cmd))
# check=True so a crash marks the Kaggle run as failed instead of passing silently.
subprocess.run(cmd, check=True, env={**os.environ, "WANDB_API_KEY": wandb_api_key})
